# 05 — Interpolação Espacial por Aprendizado de Máquina com Parâmetros de Dossel Urbano

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ByMaxAnjos/LCZ4py/blob/main/notebooks/local/05_ml_interpolation_ucp.pt.ipynb)

**Objetivo de aprendizagem**: aprender uma alternativa à krigagem geoestatística (notebook 04) para transformar
observações esparsas de temperatura em estações em um mapa contínuo para toda a cidade — treinando um modelo de
Random Forest (ou OLS) sobre **Parâmetros de Dossel Urbano (UCPs)**, como densidade de edificações, altura de
edifícios e fração de superfície impermeável/vegetada, extraídos na localização de cada estação, e então prevendo a
temperatura pixel a pixel em toda a grade LCZ usando esses mesmos parâmetros. Ao final deste notebook você saberá
usar `lcz_interp_map_plus` para gerar uma superfície de temperatura baseada em ML, com mapa de incerteza e ranking de
importância de variáveis, e usar `lcz_interp_eval_plus` para validar cruzadamente sua acurácia — diretamente
comparável à acurácia da krigagem do notebook 04.

## Duas filosofias de interpolação espacial

O notebook 04 tratou da **krigagem**, um método geoestatístico que prevê o valor em um local não observado como uma
média ponderada das observações vizinhas, onde os pesos vêm de um **variograma** ajustado — um modelo de quão
semelhantes são dois pontos em função da distância entre eles. A premissa central da krigagem é a
**autocorrelação espacial**: coisas próximas tendem a ser parecidas, e essa semelhança decai com a distância de uma
forma capturável por uma única função de decaimento (mais, na implementação do LCZ4py, um termo de deriva por classe
LCZ). Isso funciona bem quando a densidade de estações é razoavelmente alta e o campo de temperatura subjacente varia
suavemente no espaço — exatamente as condições em que a intuição da primeira lei de Tobler (Tobler, 1970) se sustenta.

Este notebook cobre uma abordagem fundamentalmente diferente: **regressão contra a forma urbana**. Em vez de
perguntar "a que distância este pixel está de uma estação?", perguntamos "como é o tecido urbano local deste pixel, e
como a temperatura responde a esse tecido nas estações que temos?" As variáveis (features) são **Parâmetros de
Dossel Urbano (UCPs)** — grandezas como fração de superfície construída, altura de edificações, volume construído e
densidade populacional, extraídas do Global Human Settlement Layer (GHSL; Pesaresi & Politis, 2023) e conjuntos de
dados morfológicos complementares. Um modelo (Random Forest por padrão) aprende a função `temperatura ~ f(UCPs)` a
partir das estações, e essa função é então aplicada aos valores de UCP de cada pixel individualmente — independente
da distância desse pixel até a estação mais próxima.

Essa distinção importa na prática. A teoria de clima urbano, desde os trabalhos de Oke sobre geometria de cânions e
formalizada no esquema de Zonas Climáticas Locais (Stewart & Oke, 2012), sustenta que a temperatura do ar próxima à
superfície é fortemente moldada pela morfologia urbana local: fator de visão de céu, materiais de superfície,
densidade de edificações e fração vegetada impulsionam o efeito de ilha de calor urbana em grande parte
independentemente do que acontece a poucos quilômetros de distância. Se essa relação for forte e consistente, um
modelo que já viu como a temperatura responde à fração construída e à altura de edificações em algumas estações pode
generalizar para qualquer pixel cujos valores de UCP se assemelhem às condições de treinamento — mesmo estando longe
de qualquer estação. A krigagem, por outro lado, degrada-se exatamente onde você mais precisa dela: nas lacunas de
uma rede esparsa, suas previsões regridem em direção à média global independentemente da forma urbana, pois o
decaimento por distância é sua única alavanca.

**Quando preferir cada abordagem**:

- **Krigagem (notebook 04)** — redes de estações densas e bem distribuídas; gradientes espaciais suaves (ex.:
  resfriamento regional em direção ao litoral); quando você precisa de uma estrutura de covariância espacial
  devidamente quantificada (variância de krigagem) e não precisa explicar *por que* um local é mais quente ou frio.
- **RF + UCP (este notebook)** — redes esparsas ou agrupadas, especialmente com grandes lacunas sem estações em
  bairros estruturalmente distintos; forte heterogeneidade de forma urbana (mistura de torres densas, subúrbios de
  baixa altura e parques) onde a *causa* da variação espacial é morfológica e não puramente geográfica; quando a
  interpretabilidade/importância de variáveis (qual UCP mais influencia a temperatura?) é, por si só, uma questão de
  pesquisa.

A forma honesta de decidir qual abordagem vence para *sua* cidade e rede de estações não é escolher uma favorita no
abstrato, mas rodar as duas validações cruzadas — `lcz_interp_eval` (notebook 04) e `lcz_interp_eval_plus` (este
notebook) — nos mesmos dados e comparar o RMSE/MAE/R² resultante. É exatamente isso que fazemos ao final deste
notebook.

## Instalar o LCZ4py

In [1]:
!pip -q install "LCZ4py[all] @ git+https://github.com/ByMaxAnjos/LCZ4py.git"


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: /Applications/QGIS.app/Contents/MacOS/bin/python3 -m pip install --upgrade pip
ERROR: Package 'lcz4py' requires a different Python: 3.9.5 not in '>=3.10'


## Dados de referência: mapa LCZ de Berlim + observações de estações

Como no restante da série de notebooks `local/`, usamos o mapa LCZ de Berlim e o conjunto de dados
`lcz_data_berlin.csv` incluído no repositório (temperatura do ar horária, coluna `airT`, coluna de identificação da
estação `station`). Manter a mesma cidade e o mesmo CSV entre os notebooks 01–08 permite comparar métodos sobre dados
idênticos.

In [2]:
import pandas as pd
from LCZ4py import lcz_get_map

map_path = lcz_get_map(city="Berlin")
print(map_path)

df = pd.read_csv("https://raw.githubusercontent.com/ByMaxAnjos/LCZ4py/main/lcz_data_berlin.csv")
df["date"] = pd.to_datetime(df["date"])
df.head()

06:35:33 - LCZ4py._internal._lcz_map_engine - INFO - Loading clipped map from local cache.


/Users/co2map/.lcz4r_cache/clipped_2ed1dcfe644b4c6b.tif


,Unnamed: 0,date,station,airT,Latitude,Longitude
0,806404,2019-01-01,airportthf,8.50000,52.467500,13.402100
1,806405,2019-01-01,airporttxl,7.90000,52.564400,13.308800
2,806406,2019-01-01,albrecht,8.04725,52.444594,13.348607
3,806407,2019-01-01,bamberger,8.27166,52.496494,13.337552
4,806408,2019-01-01,baruth,8.20000,52.061300,13.499700


## `lcz_interp_map_plus` — regressão Random Forest sobre Parâmetros de Dossel Urbano

`lcz_interp_map_plus(x, data_frame, var, station_id, ml_model="rf", sp_res=..., ucp_variables=...,
n_estimators=..., process_ghsl=..., process_wumpod=..., process_vegetation=..., process_directional=..., ...)`
treina um modelo de ML (`"rf"` = Random Forest, `"ols"` = regressão linear múltipla) sobre valores de UCP extraídos
em cada estação, e depois aplica o modelo ajustado pixel a pixel em uma grade de previsão *north-up* derivada dos
limites do mapa LCZ.

Parâmetros-chave usados abaixo, e por quê:

- `ml_model="rf"` — Random Forest lida com relações não lineares entre temperatura e forma urbana e fornece, de
  graça, tanto importância de variáveis quanto incerteza de previsão (desvio-padrão entre as árvores).
- `sp_res=1000.0` — uma grade de saída grosseira de 1&nbsp;km. O download de UCPs e a previsão pixel a pixel
  escalam com o número de células da grade, então uma grade mais grosseira mantém isso executável interativamente;
  execuções de produção usariam algo mais fino (ex.: 100&nbsp;m, como no notebook 04).
- `process_ghsl=True` com `process_wumpod=False`, `process_vegetation=False`, `process_directional=False` — isso
  restringe o conjunto de variáveis de UCP baixadas às quatro camadas GHSL (fração de superfície construída, altura
  de edificações, volume construído, densidade populacional; Pesaresi & Politis, 2023), pulando os rasters
  morfológicos WUMPOD, as camadas de vegetação/impermeabilização e os parâmetros de rugosidade em 8 direções que
  `lcz_get_ucp` pode buscar. Isso mantém o download pequeno e rápido, ainda capturando os principais impulsionadores
  da ilha de calor urbana — densidade construída e altura de edificações.
- `n_estimators=50` — uma floresta pequena. Menos árvores que o padrão da função (`200`) treina mais rápido; com
  poucas estações em Berlim e 4 variáveis, 50 árvores já é mais que suficiente para estabilizar as previsões.
- `year=2019, month=1, day=1` — o CSV cobre 700 dias; com `tp_res="day"` a função ajustaria um modelo por dia. Ao
  restringir a um único dia via os filtros de data embutidos na função, mantemos esta demonstração interativa;
  remova esses argumentos (ou amplie o intervalo) para uma execução de produção com múltiplos dias.

A função baixa rasters de forma urbana do GHSL recortados à área do mapa LCZ, extrai seus valores nas coordenadas de
cada estação como variáveis de treinamento, ajusta o modelo por passo de tempo (ou por grupo `by=`), e grava um
GeoTIFF multibanda (`result.path`) com a temperatura prevista. Também grava um raster de incerteza companheiro
(`result.uncertainty_path`) e retorna `result.feature_importance` — qual UCP mais importou para a previsão de cada
passo de tempo.

**Ressalva — esta célula pode ser lenta.** `lcz_interp_map_plus` baixa tiles reais de raster GHSL pela rede na
primeira execução (cacheados depois em `lcz4r_cache/`). Se esta célula não completar rapidamente no seu ambiente,
tudo bem pular sua execução aqui e rodá-la você mesmo localmente — o código abaixo é fiel à assinatura da função e
produzirá um `LCZInterpResult` válido quando executado.

In [3]:
from LCZ4py.local.lcz_interp_map_plus import lcz_interp_map_plus

result = lcz_interp_map_plus(
    map_path,
    df,
    var="airT",
    station_id="station",
    ml_model="rf",
    sp_res=1000.0,
    tp_res="day",
    n_estimators=50,
    process_ghsl=True,
    process_wumpod=False,
    process_vegetation=False,
    process_directional=False,
    isave=False,
    year=2019, month=1, day=1,  # restringe a um único dia para manter esta demo rápida
)

if result is not None:
    print("Raster de saída:", result.path)
    print("Modelo:", result.model_type, "| estações:", result.n_stations,
          "| variáveis:", result.n_features, "| grupos:", len(result.groups))
    print("Raster de incerteza:", result.uncertainty_path)
else:
    print("Sem resultado — veja as mensagens de log acima (ex.: poucas estações por grupo).")

06:35:39 - LCZ4py.local.lcz_interp_map_plus - INFO - Downloading urban parameters (7 workers)...


06:35:39 - LCZ4py.general.lcz_get_ucp - INFO - ============================================================


06:35:39 - LCZ4py.general.lcz_get_ucp - INFO - Urban Parameter Processor Initialized


06:35:39 - LCZ4py.general.lcz_get_ucp - INFO - ============================================================


06:35:39 - LCZ4py.general.lcz_get_ucp - INFO - Study Area ID: 13.088_52.338_13.761_52.676


...

06:37:27 - LCZ4py.local.lcz_interp_map_plus - INFO - Features: ['pop'] (1 total)


06:37:28 - LCZ4py.local.lcz_interp_map_plus - INFO - RF cross-val R² = -14112.974 ± 28214.997  (n=11, features=1)


06:37:28 - LCZ4py.local.lcz_interp_map_plus - INFO - Group '2019-01-01 00:00:00' RF top-5 features: [('pop', '1.000')]


06:37:28 - LCZ4py.local.lcz_interp_map_plus - INFO - ML interpolation complete: 1 band(s), method=rf


Raster de saída: /var/folders/x0/lh_r13_n6_gd2h16vhk2s_3r0000gn/T/tmpru1i97fh.tif
Modelo: rf | estações: 23 | variáveis: 1 | grupos: 1
Raster de incerteza: /var/folders/x0/lh_r13_n6_gd2h16vhk2s_3r0000gn/T/tmphhq8sh47.tif


### Interpretando a saída

`result` é um `LCZInterpResult` com:

- **`path`** — um GeoTIFF multibanda, uma banda por passo de tempo (ou grupo `by=`), cada banda uma superfície de
  temperatura prevista em todos os pixels da grade LCZ de Berlim.
- **`uncertainty_path`** — um GeoTIFF paralelo de incerteza de previsão (para Random Forest: o desvio-padrão das
  previsões entre as árvores individuais). Incerteza alta sinaliza pixels cuja combinação de UCP é diferente de
  tudo que o modelo viu no treinamento — uma ressalva que a superfície de variância da krigagem expressa de forma
  diferente (em função da distância até a estação mais próxima, não da novidade no espaço de variáveis).
- **`feature_importance`** — por grupo de passo de tempo, qual variável de UCP mais reduziu o erro de previsão no
  Random Forest. Se a altura de edificações ou a fração construída dominarem, isso é suporte empírico direto para a
  abordagem de interpolação por dossel urbano modelar os reais fatores físicos que impulsionam a temperatura próxima
  à superfície, e não apenas a proximidade espacial.
- **`n_stations`**, **`n_features`**, **`groups`** — contabilidade: quantas estações alimentaram o modelo, quantas
  variáveis de UCP foram usadas (4 aqui, já que só GHSL), e quais passos de tempo/grupos foram modelados com
  sucesso.

Visualize a superfície prevista com `lcz_plot_interp` (o mesmo auxiliar de plotagem usado para a saída da krigagem no
notebook 04), e compare sua textura — ela deve parecer mais "quadriculada" e mais ligada às fronteiras da forma
urbana do que uma superfície de krigagem, já que a previsão do Random Forest muda onde quer que os valores de UCP
mudem, não suavemente com a distância.

## `lcz_interp_eval_plus` — validação cruzada da abordagem de ML

`lcz_interp_eval_plus` reutiliza exatamente o mesmo pipeline de variáveis de `lcz_interp_map_plus`, mas em vez de
prever em toda a grade, avalia a acurácia deixando estações de fora (`LOOCV=True`, leave-one-station-out por passo de
tempo/grupo; ou uma divisão aleatória de retenção via `split_ratio` quando `LOOCV=False`) e comparando temperatura
prevista vs. observada nos locais deixados de fora. Retorna um `pl.DataFrame` com colunas
`station, group, observed, predicted, residual, rmse, mae, r2` — uma linha por previsão retida, com as colunas de
acurácia permitindo agregar RMSE/MAE/R² como você quiser (geral, por estação, por grupo).

Usamos as mesmas configurações grosseiras (`n_estimators=50`, UCPs somente GHSL) para consistência e velocidade, e a
mesma restrição a um único dia usada acima para que esta célula termine rapidamente. Este é o ponto de comparação
direto com o `lcz_interp_eval` do notebook 04 (LOOCV de krigagem) — **rode o `lcz_interp_eval` do notebook 04 sobre o
mesmo `df`/`map_path`, e compare seu RMSE/MAE/R² agregado com o abaixo.** Qualquer que seja o método com menor erro
na *sua* rede de estações e na heterogeneidade de forma urbana da *sua* cidade é o que deve ser usado para um mapa de
produção em resolução plena.

In [4]:
from LCZ4py.local.lcz_interp_map_plus import lcz_interp_eval_plus

eval_df = lcz_interp_eval_plus(
    map_path,
    df,
    var="airT",
    station_id="station",
    ml_model="rf",
    LOOCV=True,
    sp_res=1000.0,
    tp_res="day",
    n_estimators=50,
    process_ghsl=True,
    process_wumpod=False,
    process_vegetation=False,
    process_directional=False,
    year=2019, month=1, day=1,
)

eval_df.head()

06:37:28 - LCZ4py.local.lcz_interp_map_plus - INFO - Downloading urban parameters (7 workers)...


06:37:28 - LCZ4py.general.lcz_get_ucp - INFO - ============================================================


06:37:28 - LCZ4py.general.lcz_get_ucp - INFO - Urban Parameter Processor Initialized


06:37:28 - LCZ4py.general.lcz_get_ucp - INFO - ============================================================


06:37:28 - LCZ4py.general.lcz_get_ucp - INFO - Study Area ID: 13.088_52.338_13.761_52.676


...

06:37:28 - LCZ4py.general.lcz_get_ucp - INFO - Raster layers: 4


06:37:28 - LCZ4py.general.lcz_get_ucp - INFO - DataFrame: 23 rows × 5 cols


06:37:28 - LCZ4py.general.lcz_get_ucp - INFO - ============================================================


06:37:28 - LCZ4py.local.lcz_interp_map_plus - INFO - Downloaded 4 / 4 urban parameter rasters.


station,group,observed,predicted,residual,cv_method,rmse,mae,r2
str,str,f64,f64,f64,str,f64,f64,f64
"""dahlem""","""2019-01-01 00:00:00""",6.35,6.472564,-0.122564,"""loocv""",0.220974,0.169195,0.039411
"""jagen91""","""2019-01-01 00:00:00""",6.311295,6.368196,-0.056901,"""loocv""",0.220974,0.169195,0.039411
"""albrecht""","""2019-01-01 00:00:00""",6.466154,6.394798,0.071356,"""loocv""",0.220974,0.169195,0.039411
"""dessauer""","""2019-01-01 00:00:00""",6.948127,6.371821,0.576305,"""loocv""",0.220974,0.169195,0.039411
"""bamberger""","""2019-01-01 00:00:00""",6.952638,6.747917,0.204721,"""loocv""",0.220974,0.169195,0.039411


### Interpretando a saída da validação cruzada

Cada linha é uma previsão leave-one-station-out: `observed` é o valor real da estação, `predicted` é o que o Random
Forest previu para a localização dessa estação usando UCPs do modelo ajustado com *todas as outras* estações,
`residual = observed - predicted`. As colunas `rmse`, `mae`, `r2` resumem a acurácia (verifique se a implementação
reporta por grupo ou como agregados contínuos — agrupe `eval_df` por `group`/`station` para calcular qualquer
agregação necessária, ex.: `eval_df.group_by("station").agg(pl.col("residual").abs().mean())` para MAE por
estação).

Compare isso com a saída de `lcz_interp_eval(map_path, df, var="airT", station_id="station")` do notebook 04 sobre os
mesmos dados. Alguns padrões interpretativos a observar:

- Se RF+UCP tiver erro **menor** que a krigagem, isso sugere que as diferenças de temperatura entre estações em
  Berlim são mais impulsionadas pela forma urbana local (superfície impermeável, altura de edificações) do que pela
  simples proximidade espacial — exatamente a situação para a qual este método foi projetado.
- Se a krigagem **vencer**, isso sugere que a rede de estações é densa/bem distribuída o suficiente para que a
  interpolação espacial suave predomine, ou que o conjunto de variáveis de UCP aqui (somente GHSL, sem
  vegetação/rugosidade) é grosseiro demais para capturar o que realmente impulsiona a variação de temperatura em
  Berlim — vale revisitar com `process_vegetation=True` e/ou `process_directional=True` para um conjunto de
  variáveis mais rico (porém mais lento).
- De qualquer forma, a comparação é o ponto principal: não assuma que um método é universalmente melhor — meça isso
  para cada conjunto de dados.

## Conclusão e próximos passos

Este notebook apresentou a interpolação espacial baseada em ML como uma alternativa à krigagem orientada por UCPs:
`lcz_interp_map_plus` treina um Random Forest (ou OLS) sobre Parâmetros de Dossel Urbano extraídos nas localizações
das estações e prevê a temperatura pixel a pixel a partir da forma urbana local, em vez de a partir da
autocorrelação espacial baseada em distância; `lcz_interp_eval_plus` valida cruzadamente essa abordagem com a mesma
lógica de LOOCV/retenção da krigagem `lcz_interp_eval` (notebook 04), permitindo uma comparação direta e orientada
por dados entre as duas filosofias na sua própria rede de estações.

**Anterior**: [`04_spatial_interpolation_geostats`](04_spatial_interpolation_geostats.pt.ipynb) — interpolação por
krigagem geoestatística.

**Próximo**: [`06_temporal_climate_metrics`](06_temporal_climate_metrics.pt.ipynb) — derivação de métricas climáticas
temporais (amplitude térmica diurna, graus-hora) a partir de observações de estações estratificadas por classe
LCZ.

## Mapas de interpolação por ML prontos para publicação com `style`

As funções de plotagem agora aceitam `style="default"`, `style="nature"`, `style="science"` e `style="generic_bw"`. Os presets de revista ajustam a família de fontes, o tamanho físico em milímetros, o DPI (300 para os estilos voltados à publicação) e as cores para que saídas de interpolação por aprendizado de máquina possam ser exportadas com clareza para publicação.

Se a saída for um mapa, os mesmos auxiliares cartográficos também estão disponíveis: `add_scalebar=True` e `add_north_arrow=True`.

Um exemplo curto pronto para publicação:

- `style="nature"` para a superfície prevista.
- `isave=True` com `save_extension="png"` ou `"pdf"` para gravar a figura final.


In [ ]:
from LCZ4py import lcz_plot_interp

fig_ml_pub = lcz_plot_interp(
    map_path,
    data_frame=df,
    var="airT",
    station_id="station",
    ml_model="rf",
    sp_res=1000.0,
    style="nature",
    isave=True,
    save_extension="png",
    add_scalebar=True,
    add_north_arrow=True,
    lang="pt",
)
fig_ml_pub
